In [2]:
import json, os

racine = "/lakehouse/default/Files/bronze/aqicn"

# Trouver le fichier le plus récent, quelle que soit la date
fichiers = []
for dossier, _, noms in os.walk(racine):
    for n in noms:
        fichiers.append(os.path.join(dossier, n))

chemin = sorted(fichiers)[-1]
print(f"Fichier analysé : {chemin}\n")

with open(chemin) as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False)[:2500])

StatementMeta(, ae61dc63-a767-4dcf-b5b7-ab8a4f47922e, 4, Finished, Available, Finished, False)

Fichier analysé : /lakehouse/default/Files/bronze/aqicn/2026/09/14/toulouse_12h.json

{
  "aqi": 50,
  "idx": 3828,
  "attributions": [
    {
      "url": "https://www.atmo-occitanie.org/",
      "name": "Observatoire agréé pour assurer la surveillance de la qualité de l’air sur le territoire de la région Occitanie",
      "logo": "France-Amto.png"
    },
    {
      "url": "http://www.eea.europa.eu/themes/air/",
      "name": "European Environment Agency",
      "logo": "Europe-EEA.png"
    },
    {
      "url": "https://waqi.info/",
      "name": "World Air Quality Index Project"
    }
  ],
  "city": {
    "geo": [
      43.5873309,
      1.444026232
    ],
    "name": "Toulouse Berthelot, France",
    "url": "https://aqicn.org/city/france/midipyrenees/toulouse-berthelot",
    "location": ""
  },
  "dominentpol": "o3",
  "iaqi": {
    "dew": {
      "v": 9.5
    },
    "h": {
      "v": 25.5
    },
    "no2": {
      "v": 4.5
    },
    "o3": {
      "v": 50.2
    },
    "p": {
     

In [3]:
from pyspark.sql import functions as F

# Lecture récursive de tous les JSON Bronze AQICN
df_raw = (spark.read
    .option("multiLine", "true")
    .option("recursiveFileLookup", "true")
    .json("Files/bronze/aqicn"))

# Le chemin du fichier porte la ville et l'heure de collecte
motif = r"(\d{4})/(\d{2})/(\d{2})/([a-z]+)_(\d{2})h\.json$"
chemin = F.input_file_name()

df = (df_raw
    .withColumn("ville",   F.initcap(F.regexp_extract(chemin, motif, 4)))
    .withColumn("annee",   F.regexp_extract(chemin, motif, 1).cast("int"))
    .withColumn("mois",    F.regexp_extract(chemin, motif, 2).cast("int"))
    .withColumn("jour",    F.regexp_extract(chemin, motif, 3).cast("int"))
    .withColumn("heure",   F.regexp_extract(chemin, motif, 5).cast("int"))
)

# Horodatage de collecte : la clé métier du MERGE
df = df.withColumn(
    "horodatage_collecte",
    F.to_timestamp(
        F.format_string("%04d-%02d-%02d %02d:00:00",
                        "annee", "mois", "jour", "heure")
    )
)

silver = df.select(
    "ville",
    "horodatage_collecte",
    F.col("city.name").alias("station"),
    F.col("city.geo")[0].alias("latitude"),
    F.col("city.geo")[1].alias("longitude"),
    F.col("aqi").cast("double").alias("aqi"),
    F.col("iaqi.pm25.v").cast("double").alias("pm25_idx"),
    F.col("iaqi.pm10.v").cast("double").alias("pm10_idx"),
    F.col("iaqi.o3.v").cast("double").alias("o3_idx"),
    F.col("iaqi.no2.v").cast("double").alias("no2_idx"),
    F.col("iaqi.t.v").cast("double").alias("temperature"),
    F.col("iaqi.h.v").cast("double").alias("humidite"),
    F.col("iaqi.p.v").cast("double").alias("pression"),
    F.col("dominentpol").alias("polluant_dominant"),
    F.to_timestamp(F.col("time.iso")).alias("horodatage_station"),
    F.lit("AQICN").alias("source"),
)

print(f"{silver.count()} lignes extraites")
silver.printSchema()

StatementMeta(, ae61dc63-a767-4dcf-b5b7-ab8a4f47922e, 5, Finished, Available, Finished, False)

30 lignes extraites
root
 |-- ville: string (nullable = false)
 |-- horodatage_collecte: timestamp (nullable = true)
 |-- station: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- aqi: double (nullable = true)
 |-- pm25_idx: double (nullable = true)
 |-- pm10_idx: double (nullable = true)
 |-- o3_idx: double (nullable = true)
 |-- no2_idx: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidite: double (nullable = true)
 |-- pression: double (nullable = true)
 |-- polluant_dominant: string (nullable = true)
 |-- horodatage_station: timestamp (nullable = true)
 |-- source: string (nullable = false)



In [5]:
# Chaque règle produit un message d'erreur ou null
regles = [
    (F.col("aqi").isNull(), "aqi_manquant"),
    (~F.col("aqi").between(0, 500), "aqi_hors_plage"),
    (F.col("pm25_idx") > 999, "pm25_aberrant"),
    (F.col("pm25_idx").isNull() | F.col("pm10_idx").isNull(), "polluants_incomplets"),
    (~F.col("temperature").between(-30, 60), "temperature_hors_plage"),
    (~F.col("humidite").between(0, 100), "humidite_hors_plage"),
    (~F.col("pression").between(900, 1100), "pression_hors_plage"),
]

erreurs = F.array_compact(F.array(*[
    F.when(condition, F.lit(message)) for condition, message in regles
]))

silver = (silver
    .withColumn("erreurs", erreurs)
    .withColumn("qualite", F.when(F.size("erreurs") == 0, "OK").otherwise("WARNING"))
    .withColumn("ingere_le", F.current_timestamp())
)

silver.groupBy("qualite").count().show()
silver.select("ville", "horodatage_collecte", "aqi", "pm25_idx",
              "polluant_dominant", "qualite", "erreurs").show(10, truncate=False)

StatementMeta(, ae61dc63-a767-4dcf-b5b7-ab8a4f47922e, 7, Finished, Available, Finished, False)

+-------+-----+
|qualite|count|
+-------+-----+
|     OK|   28|
|WARNING|    2|
+-------+-----+

+-----------+-------------------+----+--------+-----------------+-------+----------------------+
|ville      |horodatage_collecte|aqi |pm25_idx|polluant_dominant|qualite|erreurs               |
+-----------+-------------------+----+--------+-----------------+-------+----------------------+
|Toulouse   |2026-09-02 13:00:00|43.0|23.0    |o3               |OK     |[]                    |
|Toulouse   |2026-09-14 12:00:00|50.0|31.0    |o3               |OK     |[]                    |
|Toulouse   |2026-09-02 10:00:00|31.0|26.0    |pm10             |OK     |[]                    |
|Bordeaux   |2026-09-02 13:00:00|32.0|21.0    |o3               |OK     |[]                    |
|Bordeaux   |2026-09-14 12:00:00|52.0|21.0    |o3               |OK     |[]                    |
|Lyon       |2026-09-02 13:00:00|17.0|NULL    |pm10             |WARNING|[polluants_incomplets]|
|Bordeaux   |2026-09-02 10:00:

In [8]:
from delta.tables import DeltaTable

TABLE = "silver_mesures_air"

# Une seule ligne par clé métier, sinon le MERGE échoue
silver_dedup = silver.dropDuplicates(["ville", "horodatage_collecte", "source"])

if not spark.catalog.tableExists(TABLE):
    (silver_dedup.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TABLE))
    print(f"Table {TABLE} créée — {silver_dedup.count()} lignes")
else:
    cible = DeltaTable.forName(spark, TABLE)
    (cible.alias("c")
        .merge(
            silver_dedup.alias("s"),
            "c.ville = s.ville AND c.horodatage_collecte = s.horodatage_collecte AND c.source = s.source"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f"MERGE effectué sur {TABLE}")

spark.table(TABLE).groupBy("qualite").count().show()
print(f"Total : {spark.table(TABLE).count()} lignes")

StatementMeta(, ae61dc63-a767-4dcf-b5b7-ab8a4f47922e, 10, Finished, Available, Finished, False)

MERGE effectué sur silver_mesures_air
+-------+-----+
|qualite|count|
+-------+-----+
|WARNING|    2|
|     OK|   28|
+-------+-----+

Total : 30 lignes


In [9]:
spark.sql(f"DESCRIBE HISTORY {TABLE}").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

StatementMeta(, ae61dc63-a767-4dcf-b5b7-ab8a4f47922e, 11, Finished, Available, Finished, False)

+-------+-----------------------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation                        |operationMetrics                                                                                                                                                                                                                  

In [10]:
spark.read.format("delta").option("versionAsOf", 0).table("silver_mesures_air").count()

StatementMeta(, ae61dc63-a767-4dcf-b5b7-ab8a4f47922e, 12, Finished, Available, Finished, False)

30

In [12]:
import os
racine = "/lakehouse/default/Tables/silver_mesures_air"
for dossier, _, fichiers in os.walk(racine):
    for f in sorted(fichiers):
        chemin = os.path.join(dossier, f)
        print(f"{os.path.getsize(chemin):>8} o  {chemin.replace(racine, '')}")

StatementMeta(, ae61dc63-a767-4dcf-b5b7-ab8a4f47922e, 14, Finished, Available, Finished, False)

In [13]:
import json, os

racine = "/lakehouse/default/Files/bronze/openweather"

fichiers = []
for dossier, _, noms in os.walk(racine):
    for n in noms:
        fichiers.append(os.path.join(dossier, n))

chemin = sorted(fichiers)[-1]
print(f"Fichier analysé : {chemin}\n")

with open(chemin) as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False))

StatementMeta(, ae61dc63-a767-4dcf-b5b7-ab8a4f47922e, 15, Finished, Available, Finished, False)

Fichier analysé : /lakehouse/default/Files/bronze/openweather/2026/09/14/toulouse_12h.json

{
  "coord": {
    "lon": 1.4437,
    "lat": 43.6043
  },
  "weather": [
    {
      "id": 800,
      "main": "Clear",
      "description": "ciel dégagé",
      "icon": "01d"
    }
  ],
  "base": "stations",
  "main": {
    "temp": 31.99,
    "feels_like": 30.15,
    "temp_min": 31.99,
    "temp_max": 34.77,
    "pressure": 1021,
    "humidity": 23,
    "sea_level": 1021,
    "grnd_level": 1000
  },
  "visibility": 10000,
  "wind": {
    "speed": 1.54,
    "deg": 0
  },
  "clouds": {
    "all": 8
  },
  "dt": 1789390140,
  "sys": {
    "type": 1,
    "id": 6467,
    "country": "FR",
    "sunrise": 1789363935,
    "sunset": 1789409266
  },
  "timezone": 7200,
  "id": 2972315,
  "name": "Toulouse",
  "cod": 200
}


In [14]:
df_meteo_raw = (spark.read
    .option("multiLine", "true")
    .option("recursiveFileLookup", "true")
    .json("Files/bronze/openweather"))

motif = r"(\d{4})/(\d{2})/(\d{2})/([a-z]+)_(\d{2})h\.json$"
chemin = F.input_file_name()

dfm = (df_meteo_raw
    .withColumn("ville", F.initcap(F.regexp_extract(chemin, motif, 4)))
    .withColumn("annee", F.regexp_extract(chemin, motif, 1).cast("int"))
    .withColumn("mois",  F.regexp_extract(chemin, motif, 2).cast("int"))
    .withColumn("jour",  F.regexp_extract(chemin, motif, 3).cast("int"))
    .withColumn("heure", F.regexp_extract(chemin, motif, 5).cast("int"))
    .withColumn("horodatage_collecte",
        F.to_timestamp(F.format_string("%04d-%02d-%02d %02d:00:00",
                                       "annee", "mois", "jour", "heure")))
)

# Colonnes parfois absentes du JSON : on les crée à NULL si besoin
for colonne in ["rain", "snow"]:
    if colonne not in dfm.columns:
        dfm = dfm.withColumn(colonne, F.lit(None).cast("struct<`1h`:double>"))

meteo = dfm.select(
    "ville",
    "horodatage_collecte",
    F.col("coord.lat").alias("latitude"),
    F.col("coord.lon").alias("longitude"),
    F.col("main.temp").cast("double").alias("temperature"),
    F.col("main.feels_like").cast("double").alias("ressenti"),
    F.col("main.humidity").cast("double").alias("humidite"),
    F.col("main.pressure").cast("double").alias("pression"),
    F.col("wind.speed").cast("double").alias("vitesse_vent"),
    F.col("wind.deg").cast("double").alias("direction_vent"),
    F.col("clouds.all").cast("double").alias("nuages_pct"),
    F.coalesce(F.col("rain.`1h`"), F.lit(0.0)).cast("double").alias("precipitation"),
    F.col("visibility").cast("double").alias("visibilite"),
    F.col("weather")[0]["description"].alias("conditions"),
    F.from_unixtime(F.col("dt")).cast("timestamp").alias("horodatage_mesure"),
    F.lit("OpenWeatherMap").alias("source"),
)

print(f"{meteo.count()} lignes extraites")
meteo.printSchema()

StatementMeta(, ae61dc63-a767-4dcf-b5b7-ab8a4f47922e, 16, Finished, Available, Finished, False)

30 lignes extraites
root
 |-- ville: string (nullable = false)
 |-- horodatage_collecte: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- ressenti: double (nullable = true)
 |-- humidite: double (nullable = true)
 |-- pression: double (nullable = true)
 |-- vitesse_vent: double (nullable = true)
 |-- direction_vent: double (nullable = true)
 |-- nuages_pct: double (nullable = true)
 |-- precipitation: double (nullable = false)
 |-- visibilite: double (nullable = true)
 |-- conditions: string (nullable = true)
 |-- horodatage_mesure: timestamp (nullable = true)
 |-- source: string (nullable = false)



In [15]:
regles_meteo = [
    (F.col("temperature").isNull(), "temperature_manquante"),
    (~F.col("temperature").between(-30, 60), "temperature_hors_plage"),
    (~F.col("humidite").between(0, 100), "humidite_hors_plage"),
    (~F.col("pression").between(900, 1100), "pression_hors_plage"),
    (F.col("vitesse_vent") < 0, "vent_negatif"),
]

erreurs_meteo = F.array_compact(F.array(*[
    F.when(cond, F.lit(msg)) for cond, msg in regles_meteo
]))

meteo = (meteo
    .withColumn("erreurs", erreurs_meteo)
    .withColumn("qualite", F.when(F.size("erreurs") == 0, "OK").otherwise("WARNING"))
    .withColumn("ingere_le", F.current_timestamp())
)

meteo.groupBy("qualite").count().show()
meteo.select("ville", "horodatage_collecte", "temperature", "humidite",
             "pression", "vitesse_vent", "precipitation", "conditions", "qualite").show(10, truncate=False)

StatementMeta(, ae61dc63-a767-4dcf-b5b7-ab8a4f47922e, 17, Finished, Available, Finished, False)

+-------+-----+
|qualite|count|
+-------+-----+
|     OK|   30|
+-------+-----+

+-----------+-------------------+-----------+--------+--------+------------+-------------+---------------------+-------+
|ville      |horodatage_collecte|temperature|humidite|pression|vitesse_vent|precipitation|conditions           |qualite|
+-----------+-------------------+-----------+--------+--------+------------+-------------+---------------------+-------+
|Lille      |2026-09-02 13:00:00|22.14      |61.0    |1021.0  |5.14        |0.39         |légère pluie         |OK     |
|Marseille  |2026-09-02 10:00:00|27.53      |41.0    |1019.0  |5.08        |0.0          |partiellement nuageux|OK     |
|Marseille  |2026-09-14 12:00:00|25.73      |48.0    |1019.0  |2.38        |0.0          |peu nuageux          |OK     |
|Marseille  |2026-09-02 13:00:00|28.98      |37.0    |1017.0  |5.97        |0.0          |peu nuageux          |OK     |
|Lyon       |2026-09-14 12:00:00|25.88      |48.0    |1022.0  |3.07     

In [16]:
TABLE_METEO = "silver_mesures_meteo"

meteo_dedup = meteo.dropDuplicates(["ville", "horodatage_collecte", "source"])

if not spark.catalog.tableExists(TABLE_METEO):
    (meteo_dedup.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TABLE_METEO))
    print(f"Table {TABLE_METEO} créée — {meteo_dedup.count()} lignes")
else:
    cible = DeltaTable.forName(spark, TABLE_METEO)
    (cible.alias("c")
        .merge(
            meteo_dedup.alias("s"),
            "c.ville = s.ville AND c.horodatage_collecte = s.horodatage_collecte AND c.source = s.source"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f"MERGE effectué sur {TABLE_METEO}")

print(f"Total : {spark.table(TABLE_METEO).count()} lignes")

StatementMeta(, ae61dc63-a767-4dcf-b5b7-ab8a4f47922e, 18, Finished, Available, Finished, False)

Table silver_mesures_meteo créée — 30 lignes
Total : 30 lignes


In [17]:
air = spark.table("silver_mesures_air")
met = spark.table("silver_mesures_meteo")

jointure = (air.alias("a")
    .join(met.alias("m"), ["ville", "horodatage_collecte"], "inner")
    .select(
        "ville", "horodatage_collecte",
        F.col("a.aqi"), F.col("a.polluant_dominant"),
        F.col("m.temperature"), F.col("m.humidite"),
        F.col("m.vitesse_vent"), F.col("m.conditions"),
    ))

print(f"Air : {air.count()} | Météo : {met.count()} | Jointure : {jointure.count()}")
jointure.orderBy(F.desc("aqi")).show(10, truncate=False)

StatementMeta(, ae61dc63-a767-4dcf-b5b7-ab8a4f47922e, 19, Finished, Available, Finished, False)

Air : 30 | Météo : 30 | Jointure : 30
+-----------+-------------------+----+-----------------+-----------+--------+------------+---------------------+
|ville      |horodatage_collecte|aqi |polluant_dominant|temperature|humidite|vitesse_vent|conditions           |
+-----------+-------------------+----+-----------------+-----------+--------+------------+---------------------+
|Lille      |2026-09-02 10:00:00|61.0|pm25             |20.36      |69.0    |4.63        |nuageux              |
|Lille      |2026-09-02 13:00:00|61.0|pm25             |22.14      |61.0    |5.14        |légère pluie         |
|Lille      |2026-09-14 12:00:00|61.0|pm25             |22.77      |78.0    |3.09        |couvert              |
|Bordeaux   |2026-09-14 12:00:00|52.0|o3               |32.15      |23.0    |1.03        |ciel dégagé          |
|Marseille  |2026-09-14 12:00:00|51.0|o3               |25.73      |48.0    |2.38        |peu nuageux          |
|Marseille  |2026-09-02 13:00:00|50.0|o3               |28